# 05 — Compare MobileNetV3-Large and EfficientNet-B0

This notebook runs the lightweight-classifier comparison used to select the
production RIDAC backbone.

## Fair-comparison rules

- Both models reuse the exact grouped split created for ResNet18.
- Validation data selects each model's F1-optimal threshold.
- Test data contains only real, non-augmented crops.
- The class-conditioned attention and FiLM head remain unchanged; only the
  visual backbone changes.

## Expected result

The completed project selected MobileNetV3-Large because it achieved the best
overall operating metrics with only 4.33 million trainable parameters.


## 1. Locate the training script and result directory

The script trains both backbones, writes each model's checkpoint and reports,
and creates one `model_comparison.csv` table containing the ResNet18 reference.


In [ ]:
# Resolve the lightweight training script and output directory
from pathlib import Path
import subprocess
import sys

# Support launching from either the RIDAC directory or its repository parent.
RIDAC_DIR = Path.cwd().resolve()
if not (RIDAC_DIR / 'train_lightweight_defect_models.py').is_file():
    candidate = RIDAC_DIR / 'misc' / 'ridac'
    if (candidate / 'train_lightweight_defect_models.py').is_file():
        RIDAC_DIR = candidate
    else:
        raise FileNotFoundError('Could not locate train_lightweight_defect_models.py')
SCRIPT = RIDAC_DIR / 'train_lightweight_defect_models.py'
OUTPUT_ROOT = RIDAC_DIR / 'object_defect_dataset' / 'balanced_augmented_full_r50_cap8_seed42' / 'lightweight_model_results'
print('Python:', sys.executable)
print('Training script:', SCRIPT)


## 2. Train both backbones or reuse the completed run

Keep `REUSE_COMPLETED_RUN=True` when the result CSV already exists. Set it to
`False` only when intentionally retraining after a data, split, architecture,
or hyperparameter change.

Training is executed in a subprocess so GPU memory and terminal progress remain
isolated from the notebook kernel.


In [ ]:
# Reuse existing metrics or launch MobileNetV3 and EfficientNet training
# The packaged project already contains completed results; avoid retraining by default.
REUSE_COMPLETED_RUN = True
comparison_path = OUTPUT_ROOT / 'model_comparison.csv'
if REUSE_COMPLETED_RUN and comparison_path.is_file():
    print('Reusing completed MobileNetV3 and EfficientNet results.')
else:
    # Use the active notebook interpreter for environment consistency.
    command = [sys.executable, str(SCRIPT)]
    completed = subprocess.run(command, cwd=RIDAC_DIR, check=False)
    if completed.returncode:
        raise RuntimeError(f'Training exited with code {completed.returncode}')


## 3. Compare overall and per-class metrics

Use more than accuracy when choosing a defect classifier. The most informative
selection columns are:

- Parameter count and completed epochs.
- Balanced accuracy.
- Defective precision and recall.
- F1 and Matthews correlation coefficient.
- ROC-AUC and PR-AUC.
- False-positive and false-negative counts.

Per-class tables identify categories that need more defect examples.


In [ ]:
# Load the completed comparison table and per-class reports
import pandas as pd
from IPython.display import display

# Every row was evaluated on the same held-out test split.
comparison = pd.read_csv(OUTPUT_ROOT / 'model_comparison.csv')
display(comparison.round(5))
for model_name in ['mobilenet_v3_large', 'efficientnet_b0']:
    print(f'\n{model_name} per-class metrics')
    metrics = pd.read_csv(OUTPUT_ROOT / model_name / 'per_class_metrics.csv')
    display(metrics.round(5))


## 4. Model-selection conclusion

For the recorded RIDAC run, MobileNetV3-Large achieved the highest accuracy,
balanced accuracy, precision, recall, F1, MCC, and ROC-AUC while remaining
smaller than both alternatives. Its checkpoint therefore powers `app.py`.

The exact production artifact is available at
`../models/mobilenet_v3_best.pt`.
